# 03 — Chronological Splitting & Sequence Building

**Goal:** Separate the dataset into a locked final test set and a development set, create expanding-window cross-validation folds, then build 3-D sliding-window tensors for sequence models.

**Modules used:** `src/data/splitters.py`, `src/data/sequence_builder.py`

---

## 0 · Imports & data setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data.load_data import load_market_data
from src.data.preprocess import apply_missing_value_policy, select_columns
from src.data.labeling import compute_forward_return, compute_threshold, make_labels
from src.data.splitters import split_dev_test, make_expanding_folds
from src.data.sequence_builder import build_sequences, drop_neutral_sequences, flatten_sequences

DATA_PATH    = ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv'
FEATURE_COLS = ['VIX_Term_Structure','Yield_Curve','SKEW_Index',
                'Risk_Appetite_Ratio','Crude_Oil','VWAP_Deviation','Volume_Momentum']
DATE_COL  = 'Date'
CLOSE_COL = 'Nasdaq_Close'
LOOKBACK  = 21   # trading days ≈ 1 calendar month
N_FOLDS   = 4
VAL_RATIO = 0.10
TEST_RATIO = 0.15

In [ ]:
# Reproduce the full preprocessing pipeline from notebooks 01-02
df_raw   = load_market_data(str(DATA_PATH), date_col=DATE_COL)
df_clean = apply_missing_value_policy(df_raw, method='ffill_then_drop_head')
df       = select_columns(df_clean, FEATURE_COLS, CLOSE_COL, DATE_COL)

print(f'Clean DataFrame: {df.shape[0]} rows, {df.shape[1]} columns')

---
## 1 · Dev / Final-Test split

The **final test set (15%)** is separated first — before any model training, threshold computation, or cross-validation. It is never used until the very last evaluation step.

```
│◄──────────── Dev Set (85%) ────────────►│◄── Test (15%) ──►│
```

In [ ]:
dev_df, test_df = split_dev_test(df, test_ratio=TEST_RATIO)

print(f'Dev  set : {len(dev_df):>5} rows  |  {dev_df[DATE_COL].iloc[0].date()} → {dev_df[DATE_COL].iloc[-1].date()}')
print(f'Test set : {len(test_df):>5} rows  |  {test_df[DATE_COL].iloc[0].date()} → {test_df[DATE_COL].iloc[-1].date()}')

### Visualise the split boundary

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3))

ax.fill_betweenx([0, 1], dev_df[DATE_COL].iloc[0], dev_df[DATE_COL].iloc[-1],
                 alpha=0.3, color='steelblue', label='Development set')
ax.fill_betweenx([0, 1], test_df[DATE_COL].iloc[0], test_df[DATE_COL].iloc[-1],
                 alpha=0.5, color='tomato', label='Final test set (locked)')

split_date = test_df[DATE_COL].iloc[0]
ax.axvline(split_date, color='black', linestyle='--', linewidth=1.5)
ax.text(split_date, 0.5, f'  {split_date.date()}', va='center', fontsize=9)

ax.set_yticks([])
ax.set_title('Dev / Test Chronological Split', fontsize=12)
ax.legend(loc='upper left')
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## 2 · Expanding-window cross-validation folds

**Expanding window strategy:**
- Validation block size is fixed.
- Each successive fold includes all previous validation data as additional training data.
- This mimics production: a model trained on all available history predicts the next unseen period.

```
Fold 1:  [Train  ──────────────] [Val]
Fold 2:  [Train  ─────────────────────] [Val]
Fold 3:  [Train  ──────────────────────────────] [Val]
Fold 4:  [Train  ─────────────────────────────────────] [Val]
```

In [ ]:
folds = make_expanding_folds(len(dev_df), n_folds=N_FOLDS, val_ratio_within_dev=VAL_RATIO)

print(f'Number of folds : {len(folds)}')
print(f'Dev set size    : {len(dev_df)} rows\n')

for i, (tr, vl) in enumerate(folds, 1):
    tr_start = dev_df[DATE_COL].iloc[tr.start].date()
    tr_end   = dev_df[DATE_COL].iloc[tr.stop - 1].date()
    vl_start = dev_df[DATE_COL].iloc[vl.start].date()
    vl_end   = dev_df[DATE_COL].iloc[vl.stop - 1].date()
    print(f'Fold {i} | Train [{tr_start} → {tr_end}] ({len(tr)} rows)  '
          f'Val [{vl_start} → {vl_end}] ({len(vl)} rows)')

### Fold diagram

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
dates = dev_df[DATE_COL]

for i, (tr, vl) in enumerate(folds):
    y = N_FOLDS - i
    ax.barh(y, len(tr),  left=tr.start,  height=0.6, color='steelblue', alpha=0.7)
    ax.barh(y, len(vl),  left=vl.start,  height=0.6, color='tomato',    alpha=0.9)
    ax.text(tr.stop / 2, y, f'Train ({len(tr)})', ha='center', va='center', fontsize=8, color='white')
    ax.text(vl.start + len(vl)/2, y, f'Val ({len(vl)})', ha='center', va='center', fontsize=8, color='white')

ax.set_xlabel('Row index within dev set')
ax.set_yticks(range(1, N_FOLDS + 1))
ax.set_yticklabels([f'Fold {N_FOLDS - i}' for i in range(N_FOLDS)])
ax.set_title('Expanding-Window Cross-Validation Folds', fontsize=12)

train_patch = mpatches.Patch(color='steelblue', alpha=0.7, label='Train')
val_patch   = mpatches.Patch(color='tomato',    alpha=0.9, label='Validation')
ax.legend(handles=[train_patch, val_patch], loc='lower right')
plt.tight_layout()
plt.show()

---
## 3 · Sequence building

For each time step `t`, we extract a window of the past `lookback=21` days:

```
X[i]  =  features[t - 20 : t + 1]   → shape (21, 7)
y[i]  =  label[t]                    → scalar {0, 1, -1}
```

The result is a **3-D tensor** of shape `(n_samples, 21, 7)`.

In [ ]:
# Compute labels using the dev-set threshold (train-fold only in real pipeline)
fwd_returns = compute_forward_return(dev_df[CLOSE_COL], horizon=1)
n_train_demo = folds[0][0].stop   # first fold train end — for demo threshold
threshold   = compute_threshold(fwd_returns.iloc[:n_train_demo], method='quantile', quantile=0.40)
labels      = make_labels(fwd_returns, threshold=threshold)

X, y, timestamps = build_sequences(dev_df, FEATURE_COLS, labels, lookback=LOOKBACK)

print(f'X shape      : {X.shape}   (n_samples, lookback, n_features)')
print(f'y shape      : {y.shape}')
print(f'Timestamp[0] : {timestamps.iloc[0].date()}  (window covers first {LOOKBACK} days)')
print(f'Timestamp[-1]: {timestamps.iloc[-1].date()}')

### Drop neutral sequences

In [ ]:
X_clean, y_clean, ts_clean = drop_neutral_sequences(X, y, timestamps, neutral_value=-1)

print(f'Before neutral removal : {X.shape[0]:>5} samples')
print(f'After  neutral removal : {X_clean.shape[0]:>5} samples')
print(f'Dropped                : {X.shape[0] - X_clean.shape[0]:>5} samples')
print(f'\nRemaining labels  →  Bull (1): {(y_clean == 1).sum()}  |  Bear (0): {(y_clean == 0).sum()}')

### Inspect one sequence

In [ ]:
sample_idx = 100
fig, axes = plt.subplots(1, len(FEATURE_COLS), figsize=(16, 3), sharey=False)
fig.suptitle(f'Sample #{sample_idx} — lookback window (21 days)  |  Label: {"Bull" if y_clean[sample_idx] == 1 else "Bear"}',
             fontsize=11)

for ax, feat, j in zip(axes, FEATURE_COLS, range(len(FEATURE_COLS))):
    ax.plot(range(LOOKBACK), X_clean[sample_idx, :, j], marker='o', markersize=2)
    ax.set_title(feat, fontsize=7)
    ax.set_xticks([])

plt.tight_layout()
plt.show()

---
## 4 · Flattening for tabular models

Logistic Regression and tree ensemble models cannot consume 3-D tensors. We flatten `(n, 21, 7)` → `(n, 147)`.

In [ ]:
X_flat = flatten_sequences(X_clean)
print(f'3D tensor shape : {X_clean.shape}')
print(f'2D flat   shape : {X_flat.shape}   ({LOOKBACK} × {len(FEATURE_COLS)} = {LOOKBACK * len(FEATURE_COLS)} features)')

---
## Summary

| Item | Value |
|---|---|
| Final test set | Last 15% of data — locked |
| CV strategy | Expanding window, 4 folds |
| Lookback window | 21 trading days |
| Sequence input shape | `(n_samples, 21, 7)` |
| Tabular input shape | `(n_samples, 147)` |

**Next:** `04_classical_ml_models.ipynb` — train Logistic Regression and tree ensemble models on the flattened sequences.